# VaR & Expected Shortfall Risk Analysis

**CQF-Level Example**: Comprehensive risk measurement:
- Historical VaR (Value at Risk)
- Parametric VaR (Gaussian)
- Monte Carlo VaR simulation
- Expected Shortfall (CVaR)
- Stressed VaR
- Component VaR attribution

**Connectors Used:**
- `qj.eod` - Portfolio holdings prices
- `qj.fred` - Market stress indicators

**API:** https://api.quantjourney.cloud

## Run Output

![27_var_expected_shortfall](../plots/27_var_expected_shortfall_output_01.png)

**Prepared by QuantJourney.** Candidate notebook source is kept clean and unexecuted. Generated run artifacts are committed under `plots/` and indexed in `plots/manifest.json`.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

from quantjourney.sdk import QuantJourneyAPI
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "png"
from scipy import stats

import os
API_KEY = os.environ.get("QJ_API_KEY", "qj_...")
qj = QuantJourneyAPI(api_key=API_KEY)
print("✓ Connected to QuantJourney API")


## 1. Define Portfolio

In [ ]:
# Portfolio holdings (typical Family Office)
portfolio = {
    'AAPL': {'shares': 500, 'sector': 'Technology'},
    'MSFT': {'shares': 400, 'sector': 'Technology'},
    'JPM': {'shares': 300, 'sector': 'Financials'},
    'JNJ': {'shares': 250, 'sector': 'Healthcare'},
    'XOM': {'shares': 400, 'sector': 'Energy'},
    'PG': {'shares': 200, 'sector': 'Consumer'},
    'GLD': {'shares': 100, 'sector': 'Commodities'},
    'TLT': {'shares': 150, 'sector': 'Bonds'},
}

symbols = list(portfolio.keys())


In [ ]:
# Fetch prices
prices = {}
for symbol in symbols:
    try:
        response = qj.eod.get_historical_prices(
            symbol=symbol,
            start_date='2020-01-01',
            end_date='2024-12-31',
            frequency='1d'
        )
        data = response.get('value', response) if isinstance(response, dict) else response
        if isinstance(data, list) and len(data) > 0:
            df = pd.DataFrame(data)
            df['date'] = pd.to_datetime(df['date'])
            df = df.set_index('date')
            prices[symbol] = df['adjusted_close' if 'adjusted_close' in df.columns else 'close']
            print(f"✓ {symbol}: {len(df)} days")
    except Exception as e:
        print(f"✗ {symbol}: {e}")

# Fallback
if len(prices) < 5:
    print("\nGenerating synthetic data...")
    dates = pd.date_range(start='2020-01-01', end='2024-12-31', freq='B')
    np.random.seed(42)
    
    initial = {'AAPL': 75, 'MSFT': 160, 'JPM': 135, 'JNJ': 145, 'XOM': 70, 'PG': 120, 'GLD': 150, 'TLT': 140}
    vols = {'AAPL': 0.30, 'MSFT': 0.28, 'JPM': 0.32, 'JNJ': 0.18, 'XOM': 0.35, 'PG': 0.15, 'GLD': 0.15, 'TLT': 0.12}
    
    for sym in symbols:
        ret = 0.10/252 + vols[sym]/np.sqrt(252) * np.random.randn(len(dates))
        prices[sym] = pd.Series(initial[sym] * np.cumprod(1 + ret), index=dates)


In [ ]:
# Combine prices and calculate returns
prices_df = pd.DataFrame(prices).dropna()
returns_df = prices_df.pct_change().dropna()

# Calculate portfolio values
current_prices = prices_df.iloc[-1]
holdings = pd.Series({sym: portfolio[sym]['shares'] for sym in symbols})
position_values = holdings * current_prices
portfolio_value = position_values.sum()

# Position weights
weights = position_values / portfolio_value

print(f"\nPortfolio Value: ${portfolio_value:,.0f}")
print(f"\nPosition Breakdown:")
for sym in symbols:
    print(f"  {sym}: ${position_values[sym]:,.0f} ({weights[sym]*100:.1f}%)")


## 2. Historical VaR

In [ ]:
# Portfolio returns
port_returns = (returns_df * weights).sum(axis=1)

# Historical VaR at different confidence levels
confidence_levels = [0.90, 0.95, 0.99]
horizons = [1, 5, 10, 21]  # days

print("Historical VaR (% of portfolio):")
print(f"{'Horizon':<10}", end='')
for cl in confidence_levels:
    print(f"VaR {cl*100:.0f}%{'':<5}", end='')
print()

var_results = {}
for horizon in horizons:
    # Rolling returns for horizon
    if horizon == 1:
        rolling_ret = port_returns
    else:
        rolling_ret = port_returns.rolling(horizon).sum().dropna()
    
    print(f"{horizon}d{'':<7}", end='')
    for cl in confidence_levels:
        var = np.percentile(rolling_ret, (1 - cl) * 100)
        var_results[(horizon, cl)] = var
        print(f"{var*100:>8.2f}%", end='')
    print()


In [ ]:
# VaR distribution visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Return Distribution', 'Historical VaR Levels']
)

# Histogram
fig.add_trace(
    go.Histogram(x=port_returns * 100, nbinsx=50, name='Returns', marker_color='cyan'),
    row=1, col=1
)

# VaR lines
colors = {'0.9': 'yellow', '0.95': 'orange', '0.99': 'red'}
for cl in confidence_levels:
    var = var_results[(1, cl)]
    fig.add_vline(x=var*100, line_dash="dash", line_color=colors[str(cl)], 
                  annotation_text=f"VaR {cl*100:.0f}%", row=1, col=1)

# VaR by horizon
for cl in confidence_levels:
    var_vals = [var_results[(h, cl)] * 100 for h in horizons]
    fig.add_trace(
        go.Scatter(x=horizons, y=[-v for v in var_vals], mode='lines+markers',
                   name=f'VaR {cl*100:.0f}%', line=dict(color=colors[str(cl)])),
        row=1, col=2
    )

fig.update_layout(
    title='Value at Risk Analysis',
    template='plotly_dark',
    height=400
)
fig.update_xaxes(title_text='Return (%)', row=1, col=1)
fig.update_xaxes(title_text='Horizon (days)', row=1, col=2)
fig.update_yaxes(title_text='VaR (%)', row=1, col=2)
fig.show()


## 3. Parametric VaR

In [ ]:
# Portfolio statistics
mean_ret = port_returns.mean()
std_ret = port_returns.std()

print(f"Portfolio Statistics (Daily):")
print(f"  Mean: {mean_ret*100:.4f}%")
print(f"  Std Dev: {std_ret*100:.4f}%")
print(f"  Skewness: {port_returns.skew():.3f}")
print(f"  Kurtosis: {port_returns.kurtosis():.3f}")

# Parametric VaR (Gaussian assumption)
print(f"\nParametric VaR (Gaussian):")
for cl in confidence_levels:
    z_score = stats.norm.ppf(1 - cl)
    var_param = mean_ret + z_score * std_ret
    var_dollar = var_param * portfolio_value
    print(f"  VaR {cl*100:.0f}%: {var_param*100:.2f}% (${abs(var_dollar):,.0f})")


In [ ]:
# Cornish-Fisher VaR (adjusting for skew and kurtosis)
def cornish_fisher_var(returns, confidence):
    """VaR adjusted for non-normality."""
    z = stats.norm.ppf(1 - confidence)
    s = returns.skew()
    k = returns.kurtosis()
    
    # Cornish-Fisher expansion
    z_cf = z + (z**2 - 1) * s / 6 + (z**3 - 3*z) * (k - 3) / 24 - (2*z**3 - 5*z) * s**2 / 36
    
    return returns.mean() + z_cf * returns.std()

print("\nCornish-Fisher VaR (adjusted for skew/kurtosis):")
for cl in confidence_levels:
    var_cf = cornish_fisher_var(port_returns, cl)
    var_hist = var_results[(1, cl)]
    print(f"  VaR {cl*100:.0f}%: {var_cf*100:.2f}% (vs Historical: {var_hist*100:.2f}%)")


## 4. Monte Carlo VaR

In [ ]:
# Monte Carlo simulation
n_simulations = 10000
horizon = 10  # days

# Covariance matrix
cov_matrix = returns_df.cov().values
mean_returns = returns_df.mean().values

# Cholesky decomposition for correlated simulation
L = np.linalg.cholesky(cov_matrix)

# Simulate paths
np.random.seed(42)
simulated_returns = np.zeros(n_simulations)

for i in range(n_simulations):
    # Generate correlated random returns for each day
    total_return = 1.0
    for d in range(horizon):
        z = np.random.randn(len(symbols))
        daily_ret = mean_returns + L @ z
        port_daily = np.dot(weights.values, daily_ret)
        total_return *= (1 + port_daily)
    simulated_returns[i] = total_return - 1

print(f"Monte Carlo Simulation ({n_simulations} paths, {horizon}-day horizon):")
for cl in confidence_levels:
    mc_var = np.percentile(simulated_returns, (1 - cl) * 100)
    mc_var_dollar = mc_var * portfolio_value
    print(f"  VaR {cl*100:.0f}%: {mc_var*100:.2f}% (${abs(mc_var_dollar):,.0f})")


In [ ]:
# Simulated distribution
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=simulated_returns * 100,
    nbinsx=100,
    name='Simulated Returns',
    marker_color='cyan',
    opacity=0.7
))

# Add VaR lines
for cl, color in zip([0.95, 0.99], ['orange', 'red']):
    var = np.percentile(simulated_returns, (1-cl)*100) * 100
    fig.add_vline(x=var, line_dash="dash", line_color=color,
                  annotation_text=f"VaR {cl*100:.0f}%: {var:.1f}%")

fig.update_layout(
    title=f'Monte Carlo VaR Distribution ({horizon}-Day Horizon)',
    xaxis_title='Portfolio Return (%)',
    yaxis_title='Frequency',
    template='plotly_dark',
    height=400
)
fig.show()


## 5. Expected Shortfall (CVaR)

In [ ]:
def expected_shortfall(returns, confidence):
    """Calculate Expected Shortfall (CVaR)."""
    var = np.percentile(returns, (1 - confidence) * 100)
    return returns[returns <= var].mean()

print("Expected Shortfall (CVaR) vs VaR:")
print(f"{'Metric':<20} {'95%':<15} {'99%':<15}")
print("-" * 50)

for cl in [0.95, 0.99]:
    var = var_results[(1, cl)]
    es = expected_shortfall(port_returns, cl)
    print(f"{'VaR':<20} {var*100:>8.2f}%{'':<5} ", end='')
print()
for cl in [0.95, 0.99]:
    es = expected_shortfall(port_returns, cl)
    print(f"{'ES/CVaR':<20} {es*100:>8.2f}%{'':<5} ", end='')
print()

# ES to VaR ratio
print(f"\nES/VaR Ratio:")
for cl in [0.95, 0.99]:
    var = var_results[(1, cl)]
    es = expected_shortfall(port_returns, cl)
    ratio = es / var if var != 0 else 0
    print(f"  {cl*100:.0f}%: {ratio:.2f}x (ES is {ratio:.0%} of VaR)")


## 6. Component VaR

In [ ]:
def component_var(returns, weights, confidence=0.95):
    """Calculate component VaR for each position."""
    port_ret = (returns * weights).sum(axis=1)
    port_var = np.percentile(port_ret, (1 - confidence) * 100)
    port_vol = port_ret.std()
    
    # Marginal VaR
    cov = returns.cov()
    marginal = cov @ weights / port_vol
    z = stats.norm.ppf(1 - confidence)
    marginal_var = marginal * z
    
    # Component VaR
    comp_var = weights * marginal_var
    
    return comp_var, marginal_var

comp_var, marg_var = component_var(returns_df, weights)

print("Component VaR Analysis (95% confidence):")
print(f"{'Asset':<10} {'Weight':<10} {'Comp VaR':<12} {'% of Total':<12}")
print("-" * 44)

total_comp_var = comp_var.sum()
for asset in symbols:
    cv = comp_var[asset]
    pct = cv / total_comp_var * 100
    print(f"{asset:<10} {weights[asset]*100:>6.1f}%   {cv*100:>8.2f}%   {pct:>8.1f}%")

print("-" * 44)
print(f"{'Total':<10} {100:>6.0f}%   {total_comp_var*100:>8.2f}%   {100:>8.1f}%")


In [ ]:
# Component VaR visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Position Weights', 'Risk Contribution (Component VaR)'],
    specs=[[{'type': 'pie'}, {'type': 'pie'}]]
)

fig.add_trace(
    go.Pie(labels=symbols, values=weights.values, hole=0.4, name='Weights'),
    row=1, col=1
)

fig.add_trace(
    go.Pie(labels=symbols, values=np.abs(comp_var.values), hole=0.4, name='Risk'),
    row=1, col=2
)

fig.update_layout(
    title='Weight vs Risk Contribution',
    template='plotly_dark',
    height=400
)
fig.show()


## 7. Stressed VaR

In [ ]:
# Identify stress periods
# COVID crash: Feb-Mar 2020
stress_periods = {
    'COVID Crash': ('2020-02-15', '2020-03-31'),
    '2022 Bear Market': ('2022-01-01', '2022-06-30'),
}

print("Stressed VaR Analysis:")
for period_name, (start, end) in stress_periods.items():
    mask = (port_returns.index >= start) & (port_returns.index <= end)
    stress_returns = port_returns[mask]
    
    if len(stress_returns) > 0:
        var_95 = np.percentile(stress_returns, 5)
        var_99 = np.percentile(stress_returns, 1)
        es_95 = expected_shortfall(stress_returns, 0.95)
        
        print(f"\n{period_name} ({start} to {end}):")
        print(f"  Days: {len(stress_returns)}")
        print(f"  VaR 95%: {var_95*100:.2f}%")
        print(f"  VaR 99%: {var_99*100:.2f}%")
        print(f"  ES 95%: {es_95*100:.2f}%")
        print(f"  Worst Day: {stress_returns.min()*100:.2f}%")


In [ ]:
# VaR breaches over time
rolling_var = port_returns.rolling(252).apply(lambda x: np.percentile(x, 5))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=port_returns.index, y=port_returns * 100,
    mode='lines', name='Daily Returns',
    line=dict(color='gray', width=0.5)
))

fig.add_trace(go.Scatter(
    x=rolling_var.index, y=rolling_var * 100,
    mode='lines', name='Rolling VaR 95%',
    line=dict(color='red', width=2)
))

# Mark breaches
breaches = port_returns[port_returns < rolling_var]
fig.add_trace(go.Scatter(
    x=breaches.index, y=breaches * 100,
    mode='markers', name='VaR Breaches',
    marker=dict(color='yellow', size=8)
))

fig.update_layout(
    title=f'VaR Breaches Over Time ({len(breaches)} breaches)',
    xaxis_title='Date',
    yaxis_title='Return (%)',
    template='plotly_dark',
    height=450
)
fig.show()


## 8. Summary Report

In [ ]:
print("="*70)
print("VaR & EXPECTED SHORTFALL RISK REPORT")
print("="*70)

print(f"\n1. PORTFOLIO SUMMARY")
print(f"   Total Value: ${portfolio_value:,.0f}")
print(f"   Holdings: {len(symbols)} positions")
print(f"   History: {len(port_returns)} days")

print(f"\n2. DAILY VaR (95% Confidence)")
var_95 = var_results[(1, 0.95)]
print(f"   Historical: {var_95*100:.2f}% (${abs(var_95*portfolio_value):,.0f})")
z = stats.norm.ppf(0.05)
var_param = mean_ret + z * std_ret
print(f"   Parametric: {var_param*100:.2f}% (${abs(var_param*portfolio_value):,.0f})")

print(f"\n3. EXPECTED SHORTFALL (95%)")
es_95 = expected_shortfall(port_returns, 0.95)
print(f"   ES: {es_95*100:.2f}% (${abs(es_95*portfolio_value):,.0f})")

print(f"\n4. TOP RISK CONTRIBUTORS")
risk_contrib = (np.abs(comp_var) / np.abs(comp_var).sum() * 100).sort_values(ascending=False)
for asset in risk_contrib.index[:3]:
    print(f"   {asset}: {risk_contrib[asset]:.1f}% of portfolio risk")

print(f"\n5. STRESS SCENARIOS")
print(f"   COVID Crash VaR 99%: ~{np.percentile(port_returns['2020-02':'2020-03'], 1)*100:.1f}%")

print("\n" + "="*70)


## Summary

This CQF-level example covered:

1. **Historical VaR**: Percentile-based risk measurement
2. **Parametric VaR**: Gaussian assumption
3. **Cornish-Fisher**: Adjusted for skew/kurtosis
4. **Monte Carlo VaR**: Simulated distribution
5. **Expected Shortfall**: Tail risk beyond VaR
6. **Component VaR**: Position-level risk attribution
7. **Stressed VaR**: Crisis period analysis

**Family Office Applications**:
- Daily risk reporting
- Position sizing decisions
- Regulatory compliance (Basel III)
- Drawdown management